<a href="https://colab.research.google.com/github/karolinakuligowska/Projektowanie_systemow_informatycznych/blob/main/Spinaker_VADER_vs_BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VADER vs BERT

In [1]:
%pip install -q pandas scikit-learn transformers sentence-transformers vaderSentiment openai accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 2.8 MB/s eta 0:00:00


In [3]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import pandas as pd

In [2]:
reviews = [
    "The delivery was fast and the product quality exceeded my expectations.",
    "Customer support ignored my complaint for five days.",
    "The mobile app is easy to use, but the payment screen crashes.",
    "The price is competitive, although delivery could be faster.",
    "I received the wrong invoice and nobody explained the issue.",
    "Premium support solved my problem in less than one hour.",
    "The checkout process failed when I tried to pay by credit card.",
    "The package arrived late and the tracking information was unclear.",
    "I was charged twice for the same order.",
    "The chatbot gave helpful answers about my return request."
]

# VADER

In [4]:
analyzer = SentimentIntensityAnalyzer()

vader_results = []

for review in reviews:
    score = analyzer.polarity_scores(review)
    vader_results.append({
        "review": review,
        "vader_compound": score["compound"],
        "vader_label": "positive" if score["compound"] >= 0.05 else "negative" if score["compound"] <= -0.05 else "neutral"
    })

In [5]:
pd.DataFrame(vader_results)

,review,vader_compound,vader_label
0,The delivery was fast and the product quality ...,0.0000,neutral
1,Customer support ignored my complaint for five...,-0.2023,negative
2,"The mobile app is easy to use, but the payment...",0.2382,positive
3,"The price is competitive, although delivery co...",0.1779,positive
4,I received the wrong invoice and nobody explai...,-0.4767,negative
5,Premium support solved my problem in less than...,0.2732,positive
6,The checkout process failed when I tried to pa...,-0.2732,negative
7,The package arrived late and the tracking info...,-0.2500,negative
8,I was charged twice for the same order.,-0.2023,negative
9,The chatbot gave helpful answers about my retu...,0.4215,positive


# BERT

explicit BERT model (avoids relying on the default Hugging Face model)

In [6]:
from transformers import pipeline
import pandas as pd

In [7]:
bert_classifier = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment"
)

bert_results = bert_classifier(reviews)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [8]:
pd.DataFrame({
    "review": reviews,
    "bert_label": [r["label"] for r in bert_results],
    "bert_score": [round(r["score"], 3) for r in bert_results]
})

,review,bert_label,bert_score
0,The delivery was fast and the product quality ...,5 stars,0.684
1,Customer support ignored my complaint for five...,1 star,0.494
2,"The mobile app is easy to use, but the payment...",3 stars,0.450
3,"The price is competitive, although delivery co...",4 stars,0.455
4,I received the wrong invoice and nobody explai...,1 star,0.762
5,Premium support solved my problem in less than...,5 stars,0.595
6,The checkout process failed when I tried to pa...,1 star,0.521
7,The package arrived late and the tracking info...,1 star,0.396
8,I was charged twice for the same order.,1 star,0.471
9,The chatbot gave helpful answers about my retu...,4 stars,0.308


# VADER vs BERT comparison

In [9]:
comparison = pd.DataFrame({
    "review": reviews,
    "vader_label": [r["vader_label"] for r in vader_results],
    "vader_compound": [r["vader_compound"] for r in vader_results],
    "bert_label": [r["label"] for r in bert_results],
    "bert_confidence": [round(r["score"], 3) for r in bert_results]
})

comparison

,review,vader_label,vader_compound,bert_label,bert_confidence
0,The delivery was fast and the product quality ...,neutral,0.0000,5 stars,0.684
1,Customer support ignored my complaint for five...,negative,-0.2023,1 star,0.494
2,"The mobile app is easy to use, but the payment...",positive,0.2382,3 stars,0.450
3,"The price is competitive, although delivery co...",positive,0.1779,4 stars,0.455
4,I received the wrong invoice and nobody explai...,negative,-0.4767,1 star,0.762
5,Premium support solved my problem in less than...,positive,0.2732,5 stars,0.595
6,The checkout process failed when I tried to pa...,negative,-0.2732,1 star,0.521
7,The package arrived late and the tracking info...,negative,-0.2500,1 star,0.396
8,I was charged twice for the same order.,negative,-0.2023,1 star,0.471
9,The chatbot gave helpful answers about my retu...,positive,0.4215,4 stars,0.308


Which model handles this sentence better? "The product is not bad."

In [10]:
new_reviews = ["The product is not bad."]

# VADER sentiment for the new review
vader_new_results = []
for review in new_reviews:
    score = analyzer.polarity_scores(review)
    vader_new_results.append({
        "review": review,
        "vader_compound": score["compound"],
        "vader_label": "positive" if score["compound"] >= 0.05 else "negative" if score["compound"] <= -0.05 else "neutral"
    })

# BERT sentiment for the new review
bert_new_results = bert_classifier(new_reviews)

# Compare results for the new review
comparison_new_review = pd.DataFrame({
    "review": new_reviews,
    "vader_label": [r["vader_label"] for r in vader_new_results],
    "vader_compound": [r["vader_compound"] for r in vader_new_results],
    "bert_label": [r["label"] for r in bert_new_results],
    "bert_confidence": [round(r["score"], 3) for r in bert_new_results]
})

print("Sentiment Analysis for 'The product is not bad.'")
comparison_new_review

Sentiment Analysis for 'The product is not bad.'


,review,vader_label,vader_compound,bert_label,bert_confidence
0,The product is not bad.,positive,0.431,3 stars,0.454


### More Negated Sentences Comparison

In [11]:
new_negated_reviews = [
    "This is not a good product.",
    "I am not unhappy with the service.",
    "The food was not bad, but not great either.",
    "It's not that I don't like it, it's just okay."
]

# VADER sentiment for new negated reviews
vader_negated_results = []
for review in new_negated_reviews:
    score = analyzer.polarity_scores(review)
    vader_negated_results.append({
        "review": review,
        "vader_compound": score["compound"],
        "vader_label": "positive" if score["compound"] >= 0.05 else "negative" if score["compound"] <= -0.05 else "neutral"
    })

# BERT sentiment for new negated reviews
bert_negated_results = bert_classifier(new_negated_reviews)

# Compare results for new negated reviews
comparison_negated_reviews = pd.DataFrame({
    "review": new_negated_reviews,
    "vader_label": [r["vader_label"] for r in vader_negated_results],
    "vader_compound": [r["vader_compound"] for r in vader_negated_results],
    "bert_label": [r["label"] for r in bert_negated_results],
    "bert_confidence": [round(r["score"], 3) for r in bert_negated_results]
})

print("Sentiment Analysis for additional negated sentences:")
display(comparison_negated_reviews)

Sentiment Analysis for additional negated sentences:


,review,vader_label,vader_compound,bert_label,bert_confidence
0,This is not a good product.,negative,-0.3412,1 star,0.567
1,I am not unhappy with the service.,positive,0.3252,3 stars,0.344
2,"The food was not bad, but not great either.",negative,-0.5448,3 stars,0.722
3,"It's not that I don't like it, it's just okay.",negative,-0.0541,3 stars,0.767


Sentiment analysis for the additional negated sentences:

"This is not a good product."

* VADER: Classified as negative with a compound score of -0.3412.
* BERT: Classified as 1 star (negative) with a confidence of 0.567.
* Both models correctly identified the negative sentiment.


"I am not unhappy with the service."

* VADER: Classified as positive with a compound score of 0.3252, successfully interpreting the double negation.
* BERT: Classified as 3 stars (neutral) with a confidence of 0.344.
* VADER seems to handle this double negation more effectively to infer a positive sentiment, while BERT leans neutral.

"The food was not bad, but not great either."

* VADER: Classified as negative with a compound score of -0.5448, indicating a slightly negative overall sentiment.
* BERT: Classified as 3 stars (neutral) with a confidence of 0.722.
* VADER captures the 'not great either' aspect more towards negative, whereas BERT sees it as neutral.

"It's not that I don't like it, it's just okay."

* VADER: Classified as negative with a compound score of -0.0541.
* BERT: Classified as 3 stars (neutral) with a confidence of 0.767.
* In this nuanced sentence, VADER registered a slight negative, while BERT's strong '3 stars' suggests a more neutral interpretation.


Overall, VADER appears to be more sensitive to negations and double negations, sometimes leading to more distinct positive or negative labels. BERT, especially for complex or somewhat ambiguous negated phrases, tends to lean towards a neutral '3 stars' rating, suggesting it might be less confident in assigning extreme sentiments to these types of sentences.

# TASK

Create 6 new review sentences:

- 2 positive
- 2 negative
- 2 containing negation, ambiguity, or mixed sentiment

Run both VADER and BERT on your sentences.

### Questions

1. In how many cases did the models agree?
2. In how many cases did they disagree?
3. Choose two sentences where the results differ:
   - Which model do you think is more accurate?
   - Why?
4. Which model handles negation better?
5. Which model would you use for analyzing thousands of customer reviews?